In [ ]:
# Cell 1: Imports and Setup
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Cell 2: Define Helper Functions and Classes
def calculate_class_weights(y_train):
    counts = np.bincount(y_train.astype(int), minlength=3)
    weights = len(y_train) / (len(counts) * counts)
    return torch.FloatTensor(weights)

class ImprovedBiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=32, num_layers=2, dropout=0.3):
        super(ImprovedBiLSTM, self).__init__()
        
        # Multi-layer LSTM with small hidden size
        self.lstm = nn.LSTM(input_size=input_size,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            batch_first=True,
                            dropout=dropout if num_layers > 1 else 0,
                            bidirectional=True)
        
        self.dropout = nn.Dropout(dropout)
        
        # Feature extraction layers
        self.fc1 = nn.Linear(hidden_size * 2, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, 3)
        
        # Initialize final layer with negative bias to counter positive predictions
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.constant_(self.fc2.bias, 0.0)  # Start with zero bias

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        # Use last timestep from both directions
        out = lstm_out[:, -1, :]
        out = self.dropout(out)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

class StockDataset(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = sequences
        self.targets = targets.astype(int)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        x = torch.FloatTensor(self.sequences[idx])
        y = torch.tensor(self.targets[idx], dtype=torch.long)   # scalar Long
        return x, y

def prepare_stock_data(df, feature_cols, seq_len=20, train_split=0.7, val_split=0.15):
    stock_data = df.sort_values('Date').reset_index(drop=True)
    
    all_sequences = []
    all_targets = []
    
    if len(stock_data) < seq_len + 1:
        raise ValueError("Not enough data to create sequences.")
        
    X_vals = stock_data[feature_cols].values
    y_vals = stock_data['Target'].values

    for i in range(seq_len, len(X_vals)):
        all_sequences.append(X_vals[i-seq_len:i])
        all_targets.append(y_vals[i])
    
    X_seq = np.array(all_sequences)
    y_seq = np.array(all_targets)
    
    # Split data
    train_idx = int(len(X_seq) * train_split)
    val_idx = int(len(X_seq) * (train_split + val_split))
    
    X_train, y_train = X_seq[:train_idx], y_seq[:train_idx]
    X_val, y_val = X_seq[train_idx:val_idx], y_seq[train_idx:val_idx]
    X_test, y_test = X_seq[val_idx:], y_seq[val_idx:]
    
    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

In [ ]:
# Cell 3: Load and Prepare Data
print("Loading and filtering data...")
df_full = pd.read_csv("../data/processed/stock_data_optimized.csv")

# Define the stock and features
STOCK_TO_USE = "AAPL"
FEATURES_TO_USE = [col for col in df_full.columns if col not in ['Date', 'Ticker', 'Target']]

# Filter data
df_filtered = df_full[df_full['Ticker'] == STOCK_TO_USE].copy()
essential_cols = FEATURES_TO_USE + ['Date', 'Ticker', 'Target']
df_final = df_filtered[essential_cols]

print(f"Data filtered for Ticker: {STOCK_TO_USE}")
print(f"Using Features: {FEATURES_TO_USE}")
print(f"Total samples: {len(df_final)}")

In [ ]:
# Cell 4: Create Sequences and Scale Data
print("\nCreating sequences...")
(X_train_unscaled, y_train), (X_val_unscaled, y_val), (X_test_unscaled, y_test) = prepare_stock_data(
    df_final,
    feature_cols=FEATURES_TO_USE,
    seq_len=20
)

print(f"Data: {len(X_train_unscaled)} train, {len(X_val_unscaled)} val, {len(X_test_unscaled)} test")
print(f"Features: {X_train_unscaled.shape[2]}")
print(f"Train class distribution: {np.bincount(y_train.astype(int))}")
print(f"Train class ratio: {np.bincount(y_train.astype(int))[1] / len(y_train):.3f}")

# Scale the data
scaler = StandardScaler()
num_samples, seq_len, num_features = X_train_unscaled.shape
X_train_reshaped = X_train_unscaled.reshape(-1, num_features)
scaler.fit(X_train_reshaped)

X_train = scaler.transform(X_train_unscaled.reshape(-1, num_features)).reshape(X_train_unscaled.shape)
X_val = scaler.transform(X_val_unscaled.reshape(-1, num_features)).reshape(X_val_unscaled.shape)
X_test = scaler.transform(X_test_unscaled.reshape(-1, num_features)).reshape(X_test_unscaled.shape)

print(f"\nX_train stats after scaling - min: {X_train.min():.3f}, max: {X_train.max():.3f}")


In [ ]:
# Cell 5: Setup Model with Class Weights
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Calculate class weights
class_weights = calculate_class_weights(y_train)
print(f"Class weights: {class_weights}")

# Create datasets
train_dataset = StockDataset(X_train, y_train)
val_dataset = StockDataset(X_val, y_val)
test_dataset = StockDataset(X_test, y_test)

# Initialize model
model = ImprovedBiLSTM(
    input_size=X_train.shape[2],
    hidden_size=32,
    num_layers=2,
    dropout=0.5
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# Optimizer with lower learning rate
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-6
)

print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
# Additional cell: Verify and fix target preparation for 3-class classification

def create_3class_target(returns, down_threshold=-0.01, up_threshold=0.01):
    """
    Create 3-class targets from returns:
    0: Down (return <= down_threshold)
    1: Neutral (down_threshold < return < up_threshold) 
    2: Up (return >= up_threshold)
    """
    targets = np.zeros(len(returns))
    targets[returns <= down_threshold] = 0  # Down
    targets[(returns > down_threshold) & (returns < up_threshold)] = 1  # Neutral
    targets[returns >= up_threshold] = 2  # Up
    return targets.astype(int)

# Check current target distribution
print("Current target distribution in your data:")
print("Unique values:", np.unique(df_final['Target']))
print("Value counts:", np.bincount(df_final['Target'].astype(int), minlength=3))

# If your current targets are binary (0,1), you may need to recreate them
# Assuming you have a returns column or can calculate it from price data
if 'Log_Return' in df_final.columns:
    # Create new 3-class targets
    new_targets = create_3class_target(df_final['Log_Return'].values)
    
    print(f"\nNew 3-class target distribution:")
    print(f"Down (0): {np.sum(new_targets == 0)}")
    print(f"Neutral (1): {np.sum(new_targets == 1)}")
    print(f"Up (2): {np.sum(new_targets == 2)}")
    
    # Update your dataframe
    df_final['Target'] = new_targets
    
    # Recreate sequences with new targets
    print("\nRecreating sequences with 3-class targets...")
    (X_train_new, y_train_new), (X_val_new, y_val_new), (X_test_new, y_test_new) = prepare_stock_data(
        df_final,
        feature_cols=FEATURES_TO_USE,
        seq_len=20
    )
    
    print(f"New train class distribution: {np.bincount(y_train_new.astype(int), minlength=3)}")
    
    # Update datasets
    train_dataset = StockDataset(X_train, y_train_new)
    val_dataset = StockDataset(X_val, y_val_new)
    test_dataset = StockDataset(X_test, y_test_new)
    
    # Recalculate class weights
    class_weights = calculate_class_weights(y_train_new)
    print(f"New class weights: {class_weights}")
    
    # Update criterion
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# Verify target format for training
print(f"\nTarget verification:")
print(f"y_train shape: {y_train.shape}")
print(f"y_train dtype: {y_train.dtype}")
print(f"y_train unique values: {np.unique(y_train)}")
print(f"y_train range: [{y_train.min()}, {y_train.max()}]")

# If targets are not integers 0,1,2, convert them
if not np.array_equal(np.unique(y_train), [0, 1, 2]):
    print("WARNING: Targets are not properly formatted as 0,1,2")
    print("Consider using the create_3class_target function above")

In [ ]:
# Cell 6: Fixed Training Loop with Debugging
train_losses, val_losses, train_accs, val_accs, learning_rates = [], [], [], [], []

num_epochs = 300
best_val_loss = float('inf')
patience_counter = 0
patience = 20

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Check batch distribution
batch_labels = []
for i, (_, labels) in enumerate(train_loader):
    batch_labels.extend(labels.numpy().flatten())
    if i >= 10:  # Check first 10 batches
        break

print(f"First 10 batches class distribution: {np.bincount(np.array(batch_labels).astype(int), minlength=3)}")
print(f"Class distribution: {np.bincount(batch_labels, minlength=3) / len(batch_labels)}")

print("\nStarting training...")
for epoch in range(num_epochs):
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    all_train_preds = []
    
    for batch_idx, (batch_X, batch_y) in enumerate(train_loader):
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        
        # Forward pass - outputs are logits for 3 classes
        logits = model(batch_X)  # Shape: [batch_size, 3]
        loss = criterion(logits, batch_y.squeeze())
        
        # Add L2 regularization on outputs
        output_reg = 0.01 * logits.pow(2).mean()
        total_loss = loss + output_reg
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        optimizer.step()
        
        train_loss += loss.item()
        
        # Get predictions using argmax for 3-class classification
        predictions = torch.argmax(logits, dim=1)  # Shape: [batch_size]
        train_correct += (predictions == batch_y.squeeze()).sum().item()
        train_total += batch_y.size(0)
        
        # Store predictions for distribution analysis
        all_train_preds.extend(predictions.cpu().numpy())
        
        # Debug first batch of first few epochs
        if epoch < 5 and batch_idx == 0:
            probs = torch.softmax(logits, dim=1)
            print(f"Epoch {epoch+1}, Batch 1 - logits range: [{logits.min():.2f}, {logits.max():.2f}]")
            print(f"Predictions distribution: {np.bincount(predictions.cpu().numpy(), minlength=3)}")
    
    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            logits = model(batch_X)  # [batch_size, 3]
            loss = criterion(logits, batch_y.squeeze())
            val_loss += loss.item()
            
            # Get predictions using argmax
            predictions = torch.argmax(logits, dim=1)  # [batch_size]
            val_correct += (predictions == batch_y.squeeze()).sum().item()
            val_total += batch_y.size(0)
    
    # Calculate metrics
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    current_lr = optimizer.param_groups[0]['lr']
    
    # Store metrics
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    learning_rates.append(current_lr)
    
    # Print progress
    if epoch % 5 == 0:
        pred_dist = np.bincount(np.array(all_train_preds).astype(int), minlength=3)
        print(f"Epoch {epoch+1}: Loss={avg_train_loss:.4f}/{avg_val_loss:.4f}, "
              f"Acc={train_acc:.3f}/{val_acc:.3f}, Preds={pred_dist}")
    
    scheduler.step(avg_val_loss)
    
    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model_fixed.pth')
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

print(f"\nBest validation loss: {best_val_loss:.4f}")

In [ ]:
# Cell 7: Fixed Evaluation
print("\nEvaluating on test set...")
model.load_state_dict(torch.load('best_model_fixed.pth'))
model.eval()

all_predictions, all_targets, all_probs = [], [], []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        logits = model(batch_X)  # Shape: [batch_size, 3]
        
        # Get class probabilities using softmax
        probs = torch.softmax(logits, dim=1)  # Shape: [batch_size, 3]
        
        # Get predictions using argmax
        predictions = torch.argmax(logits, dim=1)  # Shape: [batch_size]
        
        # Store results
        all_predictions.extend(predictions.cpu().numpy())
        all_targets.extend(batch_y.squeeze().cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# Convert to numpy arrays
all_predictions = np.array(all_predictions)
all_targets = np.array(all_targets)
all_probs = np.array(all_probs)

# Calculate metrics
accuracy = accuracy_score(all_targets, all_predictions)
print(f"Test Accuracy: {accuracy:.3f}")

print("\nClassification Report:")
print(classification_report(all_targets, all_predictions,
                           target_names=['Down', 'Neutral', 'Up'], 
                           labels=[0, 1, 2]))

# Confusion Matrix
cm = confusion_matrix(all_targets, all_predictions, labels=[0, 1, 2])
print(f"\nConfusion Matrix:")
print("Predicted:  Down  Neutral  Up")
for i, actual in enumerate(['Down', 'Neutral', 'Up']):
    print(f"Actual {actual:>7}: {cm[i]}")

# Class distribution analysis
print(f"\nActual class distribution: {np.bincount(all_targets, minlength=3)}")
print(f"Predicted class distribution: {np.bincount(all_predictions, minlength=3)}")

# Confidence analysis
print(f"\nPrediction confidence analysis:")
for class_idx, class_name in enumerate(['Down', 'Neutral', 'Up']):
    class_mask = all_predictions == class_idx
    if np.sum(class_mask) > 0:
        avg_confidence = np.mean(all_probs[class_mask, class_idx])
        print(f"{class_name}: {avg_confidence:.3f} average confidence")

# Per-class accuracy
print(f"\nPer-class accuracy:")
for class_idx, class_name in enumerate(['Down', 'Neutral', 'Up']):
    class_mask = all_targets == class_idx
    if np.sum(class_mask) > 0:
        class_acc = np.mean(all_predictions[class_mask] == all_targets[class_mask])
        print(f"{class_name}: {class_acc:.3f}")
    else:
        print(f"{class_name}: No samples in test set")

In [ ]:
# Cell 8: Fixed Visualization for 3-Class Classification
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'3-Class LSTM Results for {STOCK_TO_USE}', fontsize=16)

# Training curves
epochs = range(1, len(train_losses) + 1)
axes[0,0].plot(epochs, train_losses, 'b-', label='Train')
axes[0,0].plot(epochs, val_losses, 'r-', label='Validation')
axes[0,0].set_title('Loss Curves')
axes[0,0].legend()
axes[0,0].grid(True)

# Accuracy curves
axes[0,1].plot(epochs, train_accs, 'b-', label='Train')
axes[0,1].plot(epochs, val_accs, 'r-', label='Validation')
axes[0,1].set_title('Accuracy Curves')
axes[0,1].legend()
axes[0,1].grid(True)

# Learning rate
axes[0,2].plot(epochs, learning_rates, 'g-')
axes[0,2].set_title('Learning Rate')
axes[0,2].set_yscale('log')
axes[0,2].grid(True)

# Confusion matrix (3x3 for 3 classes)
cm = confusion_matrix(all_targets, all_predictions, labels=[0, 1, 2])
sns.heatmap(cm, annot=True, fmt='d', ax=axes[1,0], 
           xticklabels=['Down', 'Neutral', 'Up'], 
           yticklabels=['Down', 'Neutral', 'Up'],
           cmap='Blues')
axes[1,0].set_title('Confusion Matrix')
axes[1,0].set_xlabel('Predicted')
axes[1,0].set_ylabel('Actual')

# Class probability distributions
class_names = ['Down', 'Neutral', 'Up']
colors = ['red', 'gray', 'green']

for i, (class_name, color) in enumerate(zip(class_names, colors)):
    class_probs = all_probs[:, i]
    axes[1,1].hist(class_probs, bins=30, alpha=0.6, label=f'{class_name}', 
                   color=color, density=True)

axes[1,1].set_title('Class Probability Distributions')
axes[1,1].set_xlabel('Probability')
axes[1,1].set_ylabel('Density')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

# Class prediction distribution comparison
x = np.arange(3)
width = 0.35
actual_counts = np.bincount(all_targets, minlength=3)
predicted_counts = np.bincount(all_predictions, minlength=3)

axes[1,2].bar(x - width/2, actual_counts, width, label='Actual', alpha=0.8)
axes[1,2].bar(x + width/2, predicted_counts, width, label='Predicted', alpha=0.8)
axes[1,2].set_title('Class Distribution Comparison')
axes[1,2].set_xlabel('Class')
axes[1,2].set_ylabel('Count')
axes[1,2].set_xticks(x)
axes[1,2].set_xticklabels(['Down', 'Neutral', 'Up'])
axes[1,2].legend()
axes[1,2].grid(True, alpha=0.3)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Additional analysis: prediction confidence by class
print("\nPrediction Analysis:")
print("="*50)

# Show some example predictions with confidence
print("\nSample predictions with confidence:")
sample_indices = np.random.choice(len(all_predictions), size=10, replace=False)

for idx in sample_indices:
    actual = all_targets[idx]
    predicted = all_predictions[idx]
    confidence = all_probs[idx, predicted]
    
    actual_name = class_names[actual]
    predicted_name = class_names[predicted]
    correct = "✓" if actual == predicted else "✗"
    
    print(f"{correct} Actual: {actual_name:>7}, Predicted: {predicted_name:>7}, "
          f"Confidence: {confidence:.3f}")

# Calculate and display per-class metrics
print(f"\nDetailed Per-Class Metrics:")
print("="*50)

for i, class_name in enumerate(class_names):
    # Precision: TP / (TP + FP)
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"{class_name}:")
    print(f"  Precision: {precision:.3f}")
    print(f"  Recall: {recall:.3f}")
    print(f"  F1-Score: {f1:.3f}")
    print(f"  Support: {cm[i, :].sum()}")
    print()

In [ ]:
# Check if your target is predictable at all
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train.reshape(len(X_train), -1), y_train)
print(f"RF accuracy: {rf.score(X_test.reshape(len(X_test), -1), y_test):.3f}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Make sure your DataFrame has all features and the target
feature_cols = [col for col in df_final.columns if col not in ['Date', 'Ticker', 'Target']]
corr_matrix = df_final[feature_cols + ['Target']].corr()


In [ ]:
plt.figure(figsize=(8, 12))
# Only show correlation of each feature with Target (excluding Target itself)
sns.barplot(
    y=feature_cols,
    x=corr_matrix['Target'].loc[feature_cols].abs().sort_values(ascending=False),
    orient='h'
)
plt.title('Absolute Feature Correlation with Target')
plt.xlabel('Correlation')
plt.show()


In [ ]:
plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Prepare data for RF: flatten sequences if using sequence model
X_rf = X_train.reshape(len(X_train), -1)
y_rf = y_train

# If using time-series, usually take the last step's features for each sample
# Or: flatten the time window, as above

rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_rf, y_rf)

importances = rf.feature_importances_

# Map importances to feature names (expand feature names if flattened)
if X_rf.shape[1] == len(feature_cols):
    names = feature_cols
else:
    names = [f"{col}_t{t}" for t in range(X_train.shape[1]) for col in feature_cols]

# Show top 20 features
import pandas as pd
imp_df = pd.DataFrame({'feature': names, 'importance': importances})
imp_df = imp_df.sort_values(by='importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(y='feature', x='importance', data=imp_df.head(20))
plt.title('Top 20 Feature Importances (Random Forest)')
plt.show()
